In [18]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import text, create_engine, bindparam
import os
from dotenv import load_dotenv
import re
from datetime import datetime
load_dotenv()
DB_HOST = os.getenv("DB_HOST")
engine = create_engine(DB_HOST)
SUBSTATION_ID = ["CIPC0001"]

# PRIMARY LINES

In [9]:
stmt = text('''
            select pl.*, b.feeder_id,
            cw.type as conductor_type,
            cw.size as conductor_size,
            cw.unit as conductor_unit,
            cw.strand as conductor_strand,
            nc.type as neutral_type,
            nc.size as neutral_size,
            nc.unit as neutral_unit
            from gis.primary_lines as pl
            left join gis.bus as b
            on b.bus_id = pl.from_bus_id
            left join gis.conductor_wires as cw
            on cw.id = pl.conductor_wire
            left join gis.neutral_concentric_cable as nc
            on pl.neutral_wire = nc.id
            where b.substation_id in :substation
            ''').bindparams(bindparam("substation", expanding=True))
pl_df = gpd.read_postgis(stmt, engine, geom_col="geom", params={
    "substation": SUBSTATION_ID})
pl_df.drop(columns=["feeder_id"], inplace=True)
pl_df.explore()

# BUS

In [10]:
stmt = text("""
            select * from gis.bus
            where substation_id in :substation
            """).bindparams(bindparam("substation", expanding=True))
bus_df = gpd.read_postgis(stmt, engine, geom_col='geom', params={"substation": SUBSTATION_ID})
bus_df = bus_df[['bus_id','description','nominal_voltage_kv', 'geom']]
bus_df.explore()

# DISTRIBUTION TRANSFORMER

In [14]:
stmt = text(
    """
    select dt.*, tt.kva_rating, tt.primary_voltage_rating, secondary_voltage_rating from gis.distribution_transformer as dt 
    left join gis.bus as b
    on  b.bus_id = dt.from_primary_bus_id
    left join gis.transformer_type as tt 
    on tt.id = dt.transformer_type
    where b.substation_id in :substation
    
    """
).bindparams(bindparam("substation", expanding=True))
dt_df = gpd.read_postgis(stmt, engine, geom_col='geom', params={"substation": SUBSTATION_ID})


cor_bus_dt_df = pd.read_excel("./sample-data/cor_bus_202401.xls", sheet_name="Distribution Transformer", engine="calamine")

to_drop_columns = [col for col in cor_bus_dt_df.columns if re.search("Unnamed", col)]
unique_values_df = cor_bus_dt_df[
        [
            'KVA Rating',
            'Secondary Voltage Rating (kV)',
            'Primary Voltage Rating(kV)',
            'X/R Ratio',
            'No-Load Loss (kW)',
            'Exciting Current (%)'
        ]
    ]

unique_values_df = unique_values_df.drop_duplicates()
dt_df['kva_rating'] = dt_df['kva_rating'].apply(lambda x: x.replace("kVA", "")).astype("float")
dt_merge = dt_df.merge(unique_values_df, how="left", 
            left_on=["kva_rating", 'primary_voltage_rating', 'secondary_voltage_rating'], 
            right_on=["KVA Rating", 'Primary Voltage Rating(kV)', 'Secondary Voltage Rating (kV)'])
dt_merge.drop(columns=['Primary Voltage Rating(kV)', 'Secondary Voltage Rating (kV)','remarks','image','is_active'], inplace=True)
dt_clean = dt_merge
dt_clean.explore()

# CONSUMER METER

In [124]:
stmt = text("""
            select cm.* from gis.consumer_meter as cm
            join gis.service_drop as sd
            on cm.account_no = sd.to_customer_id
            join gis.bus as b
            on b.bus_id = sd.from_bus_id
            where b.substation_id IN :substation
            """).bindparams(
    bindparam("substation", expanding=True)
)
cm_df = gpd.read_postgis(stmt, engine, geom_col='geom', params={"substation": SUBSTATION_ID})

# LUMP DATA
lump_cm_df = cm_df[cm_df['account_type'] == 'Consumer Lump']
lump_cm_df.sort_values(by='account_no', inplace=True)


In [140]:
BILLING_PERIOD = '202502'
kwh_df = pd.read_csv("./kwh-data/kwh_2025.csv")
kwh_df['billing_period'] =  kwh_df['month'].apply(lambda x: datetime.strptime(f"{x} 2025", "%B %Y").strftime("%Y%m"))
kwh_df['account_no'] = kwh_df['account_no'].str.replace("-", "", regex=False).str.strip().astype("string")
kwh_df = kwh_df[kwh_df['billing_period'] == BILLING_PERIOD]

# KWH DATA WITH GEOMETRY
kwh_df_with_geom = kwh_df.merge(cm_df, how="inner", left_on="account_no", right_on="account_no")
kwh_df_with_geom = kwh_df_with_geom[['account_no', 'account_name_x', 'kwh_used', 'billing_period', 'type']]
kwh_df_with_geom.rename(columns={
    'account_no' : 'customer_id',
    'account_name_x': 'customer_name',
    'type' : 'customer_type',   
},inplace=True)
kwh_df_with_geom['customer_id'].apply(lambda x: x[:6] + "0000").unique()
kwh_df_with_geom_account = kwh_df_with_geom['customer_id']

# KWH LUMP DATA
kwh_df_without_geom = kwh_df.loc[~kwh_df['account_no'].isin(kwh_df_with_geom['customer_id'])]


kwh_df_without_geom['lump_account_no'] = kwh_df_without_geom['account_no'].apply(lambda x: x[:6] + "0000")
lump_kwh_filter = lump_cm_df.merge(kwh_df_without_geom, how="inner", left_on="account_no", right_on="lump_account_no")
lump_kwh_filter = lump_kwh_filter.groupby(["lump_account_no", "account_name_x", "account_type", "billing_period"]).agg({
    "kwh_used" : "sum"
}).reset_index()



lump_kwh_filter.rename(columns={
    'lump_account_no' : 'customer_id',
    'account_name_x' : 'customer_name',
    'account_type' : 'customer_type'
},inplace=True)

lump_kwh_filter

clean_kwh_df = pd.concat([kwh_df_with_geom, lump_kwh_filter])
clean_kwh_df['kwh_used'].sum()


np.float64(3713548.0)